# Compute Sentence-Transformer Embeddings — iGEM Teams

Reads the **iGEM Teams** dataset, concatenates title + abstract for each
record, computes embeddings with a sentence-transformer model, and saves the
results (embedding matrix + aligned corpus) to today's run folder,
`assets/<date>/02/`.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# and setup_run.py reside; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd

from setup_run import setup
from aux.embeddings import prepare_text, encode_texts, save_embeddings

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
RUN             = setup(corpus="teams")  # today's run folder: assets/<date>/02/
RAW_FILE        = "00/igem.txt"       # source dataset: <stage>/<file>
TITLE_COL       = "TI"
ABSTRACT_COL    = "AB"
ID_COL          = "UT"
EMBEDDINGS_FILE = "teams_embeddings.npy"
CORPUS_FILE     = "teams_corpus.txt"

## 1. Load dataset

In [3]:
teams = pd.read_csv(RUN.get(RAW_FILE), sep="\t")
print(f"iGEM Teams: {len(teams):,} rows")

iGEM Teams: 3,811 rows


## 2. Prepare text for embedding

Concatenate title and abstract into a single `text` field, clean minimally
(keep alphabetic characters, collapse whitespace), and drop rows with no text.

In [4]:
corpus = prepare_text(teams, title_col=TITLE_COL, abstract_col=ABSTRACT_COL)
print(f"Corpus: {len(corpus):,} docs")

Corpus: 3,811 docs


## 3. Compute embeddings

In [5]:
embeddings = encode_texts(corpus["text"].tolist())
print(f"shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


Batches:   0%|          | 0/120 [00:00<?, ?it/s]

shape: (3811, 384)


## 4. Save embeddings and corpus metadata

In [6]:
save_embeddings(RUN, embeddings, corpus, ID_COL, EMBEDDINGS_FILE, CORPUS_FILE)
print(f"Saved {len(corpus):,} docs → {RUN.dir}")
print(f"  {EMBEDDINGS_FILE}")
print(f"  {CORPUS_FILE}")

Saved 3,811 docs → /Users/cristian/Desktop/GitHub/igem-synbio/assets/embeddings
  teams_embeddings.npy
  teams_corpus.txt
